# Mock Exam: Data Leakage Detection (Interactive Version)

## 🎯 Your Mission
You've trained a model with **99.8% accuracy** - but your manager is suspicious!

**Your task:**
1. Run the code below and observe the "too good to be true" results
2. Find ALL data leakage issues (there are 7 major bugs)
3. Fix them step by step
4. Achieve realistic performance (~75-80% accuracy)

## 📝 Instructions
1. First, generate the dataset by running the "Setup" cell
2. Then run the "Leaky Model" cells and observe the results
3. Try to identify what's wrong before looking at hints
4. Fix the issues in the "Your Fixed Version" section

**Time limit:** 45 minutes  
**Don't peek at solution.py until you've tried!**

## Setup: Generate Dataset

In [1]:
# Run this first to generate the dataset
%run generate_dataset.py

Generating dataset with intentional leakage issues...
Original dataset shape: (20640, 9)
Columns: ['median_income', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'latitude', 'longitude', 'median_house_value']
Target distribution:
high_value
0    10323
1    10317
Name: count, dtype: int64

✅ Dataset saved to: housing_with_target_leakage.csv
Final shape: (20640, 9)

Columns: ['median_income', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'ocean_proximity', 'median_house_value', 'high_value']

Missing values:
median_income          411
housing_median_age     391
total_rooms              0
total_bedrooms        1023
population               0
households               0
ocean_proximity          0
median_house_value       0
high_value               0
dtype: int64

⚠️ WARNING: This dataset contains 'median_house_value' which leaks target information!
This is intentional for the mock exam.


## Part 1: The Leaky Model (Run and Observe)

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

# Load data
print("Loading data...")
df = pd.read_csv("housing_with_target_leakage.csv")
print(f"Dataset shape: {df.shape}")
df.head()

Loading data...
Dataset shape: (20640, 9)


,median_income,housing_median_age,total_rooms,total_bedrooms,population,households,ocean_proximity,median_house_value,high_value
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,NEAR OCEAN,4.526,1
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,INLAND,3.585,1
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,INLAND,3.521,1
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,<1H OCEAN,3.413,1
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,NEAR BAY,3.422,1


### 🐛 BUG ZONE #1: Feature Engineering

**Question:** What's wrong with this code?

In [3]:
# Creating features using statistics from ENTIRE dataset
print("Engineering features...")

# BUG: Using df.mean() and df.std() on full dataset!
df['price_vs_mean'] = df['median_house_value'] / df['median_house_value'].mean()
df['price_zscore'] = (df['median_house_value'] - df['median_house_value'].mean()) / df['median_house_value'].std()

# BUG: Target encoding on full dataset!
ocean_proximity_means = df.groupby('ocean_proximity')['high_value'].mean()
df['ocean_proximity_target_encoded'] = df['ocean_proximity'].map(ocean_proximity_means)

# Safe features (row-level operations)
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

print(f"Features created. New shape: {df.shape}")

# YOUR ANSWER: What are the bugs here?
# Bug 1: _______________________________
# Bug 2: _______________________________

Engineering features...
Features created. New shape: (20640, 15)


### 🐛 BUG ZONE #2: Imputation and Scaling

**Question:** When should we split the data?

In [4]:
# BUG: Imputing BEFORE split!
print("Handling missing values...")
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
if len(cat_cols) > 0:
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# BUG: Scaling BEFORE split!
print("Scaling features...")
scaler = StandardScaler()
numeric_features = ['median_income', 'housing_median_age', 'total_rooms',
                   'total_bedrooms', 'population', 'households',
                   'rooms_per_household', 'bedrooms_per_room', 
                   'population_per_household', 'price_vs_mean', 'price_zscore']

df[numeric_features] = scaler.fit_transform(df[numeric_features])

# YOUR ANSWER: What's the problem with doing imputation and scaling here?
# Answer: _______________________________

Handling missing values...
Scaling features...


### 🐛 BUG ZONE #3: Features and Splitting

**Question:** Are all these features legitimate?

In [5]:
# BUG: Including features that leak target information!
feature_cols = [col for col in df.columns if col not in ['high_value']]
X = df[feature_cols]
y = df['high_value']

print(f"Features ({len(feature_cols)}):")
print(feature_cols)
print("\n⚠️ Notice: median_house_value is included!")
print("⚠️ Notice: price_vs_mean and price_zscore are derived from median_house_value!")
print("⚠️ Notice: ocean_proximity_target_encoded uses the target variable!")

# YOUR ANSWER: Which features should be removed and why?
# Features to remove: _______________________________
# Reason: _______________________________

Features (14):
['median_income', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'ocean_proximity', 'median_house_value', 'price_vs_mean', 'price_zscore', 'ocean_proximity_target_encoded', 'rooms_per_household', 'bedrooms_per_room', 'population_per_household']

⚠️ Notice: median_house_value is included!
⚠️ Notice: price_vs_mean and price_zscore are derived from median_house_value!
⚠️ Notice: ocean_proximity_target_encoded uses the target variable!


In [6]:
# Finally splitting (way too late!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (16512, 14)
Test set: (4128, 14)


In [7]:
# Encoding (already leaked via target encoding earlier)
if 'ocean_proximity' in X_train.columns:
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    
    encoded_train = encoder.fit_transform(X_train[['ocean_proximity']])
    encoded_test = encoder.transform(X_test[['ocean_proximity']])
    
    encoded_cols = encoder.get_feature_names_out(['ocean_proximity'])
    
    X_train_encoded = pd.DataFrame(encoded_train, columns=encoded_cols, index=X_train.index)
    X_test_encoded = pd.DataFrame(encoded_test, columns=encoded_cols, index=X_test.index)
    
    X_train = pd.concat([X_train.drop('ocean_proximity', axis=1), X_train_encoded], axis=1)
    X_test = pd.concat([X_test.drop('ocean_proximity', axis=1), X_test_encoded], axis=1)

### Train and Evaluate (Observe the Suspiciously High Accuracy)

In [8]:
# Train model
print("Training model...")
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print("🚨 RESULTS (TOO GOOD TO BE TRUE!) 🚨")
print("="*60)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Training model...

🚨 RESULTS (TOO GOOD TO BE TRUE!) 🚨
Training Accuracy: 1.0000
Test Accuracy: 1.0000
CV Accuracy: 1.0000 (+/- 0.0000)

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      2065
         1.0       1.00      1.00      1.00      2063

    accuracy                           1.00      4128
   macro avg       1.00      1.00      1.00      4128
weighted avg       1.00      1.00      1.00      4128



In [9]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))
print("\n⚠️ Notice which features are most important!")


Top 10 Most Important Features:
                feature  importance
6    median_house_value    0.352167
8          price_zscore    0.319282
7         price_vs_mean    0.256607
0         median_income    0.044561
11    bedrooms_per_room    0.007743
5            households    0.007450
10  rooms_per_household    0.006242
2           total_rooms    0.004169
1    housing_median_age    0.001002
3        total_bedrooms    0.000548

⚠️ Notice which features are most important!


## Part 2: Your Turn - Fix the Issues!

### 📝 Before you start coding:

**List all the bugs you found:**

1. Bug: _______________________________
2. Bug: _______________________________
3. Bug: _______________________________
4. Bug: _______________________________
5. Bug: _______________________________
6. Bug: _______________________________
7. Bug: _______________________________

### Now implement your fixes below:

In [ ]:
# YOUR FIXED VERSION HERE
# Start fresh - reload the data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

np.random.seed(42)

# Load data
df = pd.read_csv("housing_with_target_leakage.csv")

# TODO: Fix #1 - Remove leaky features
# Your code here:


# TODO: Fix #2 - Split data FIRST
# Your code here:


# TODO: Fix #3 - Build proper pipeline
# Your code here:


# TODO: Fix #4 - Train and evaluate properly
# Your code here:

## Part 3: Verify Your Fixes

After implementing your fixes, answer these questions:

1. **What is your new test accuracy?** _______  
   (Should be around 75-80%)

2. **How does it compare to the leaky model?**  
   _______________________________

3. **Are train and test accuracy close?**  
   _______________________________

4. **Which features are now most important?**  
   _______________________________

5. **What was the biggest source of leakage?**  
   _______________________________

## Part 4: Reflection

### Key Takeaways

Write down the most important lessons you learned:

1. _______________________________
2. _______________________________
3. _______________________________

### Real-World Application

How will you prevent data leakage in your future projects?

_______________________________
_______________________________
_______________________________

---

## 🎓 Check Your Solution

When you're done, compare with the solution:

```python
%run solution.py
```

Or read:
- `ANSWER_KEY.md` for detailed explanations
- `hints.md` if you're stuck